# 05b · Folder-grouped development evaluation of the classical baselines

## Purpose
The closed-set validation split saturates (notebook 05): several models classify it perfectly, so it cannot rank models
or show what generalises. This notebook evaluates the same classical models on a **development pool** with folds grouped
by **folder proxy** (class × severity folder, a proxy for an acquisition session), so every recording is predicted by a
model that never saw its folder (normal excepted, see below).

**Data rule.** The pool excludes every reserved test: `split_closed` test, `split_severity` test and `split_speed_block`
test. Known fault classes are used here, which is allowed for this closed-set development exercise; it also means the
classes later withheld in LOCO have influenced method development (see Limitations).

## Inputs
`results/baselines/dev_pool_inventory.csv`, `results/dev_folds.csv`, `results/baselines/dev_eval.json` and
`results/scores/dev_r1_<model>.parquet`, written by `scripts/05_baselines.py` (tasks `dev-inventory`, `dev-folds`, `dev`).
The model configuration and recording-level aggregation are identical to Stage 2B.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the settings, <code>metadata.csv</code>, the development results and the fold assignment, and stops if the results were produced under different split assignments than the current ones.</div>

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml.cache import check_split_hash
from ml.config import load_config
from ml.labels import CLASS6, CLASS10
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR

cfg = load_config(DEFAULT_CONFIG)
meta = pd.read_csv(SYSTEM_CODE_DIR / "metadata.csv")
RES = SYSTEM_CODE_DIR / "results" / "baselines"
FIG = SYSTEM_CODE_DIR / "results" / "figures"
SCORES = SYSTEM_CODE_DIR / "results" / "scores"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
dev = json.loads((RES / "dev_eval.json").read_text())
check_split_hash(dev["provenance"]["split_hash"], meta, "dev_eval.json")
folds = pd.read_csv(SYSTEM_CODE_DIR / "results" / cfg.development.folds_file)
K = dev["provenance"]["k"]
print(f"{dev['provenance']['feature_table']}; pool {dev['provenance']['n_recordings']} recordings, K = {K}")
print(dev["provenance"]["bootstrap"])

## 1. The development pool and the folds
Pool rule (config `development`): `split_closed` ∈ {train, val} AND `split_severity` ≠ test AND `split_speed_block` ≠ test.
K = the minimum number of eligible folder proxies over **fault** classes. Fault folders go to folds within each class
(largest first, to the fold with the fewest recordings of that class); a folder is never split. **Normal is the
exception**: it is a single folder, so its recordings are dealt to the folds individually, stratified by speed, and normal
is evaluated **within-folder**.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows the pool inventory (recordings per folder proxy, with their speed range), then the fold assignment: recordings per class in each fold, and which folder proxies each fold holds.</div>

In [ ]:
inv = pd.read_csv(RES / "dev_pool_inventory.csv")
print(f"{inv.recordings.sum()} recordings, speeds {inv.speed_min_hz.min():.1f}–{inv.speed_max_hz.max():.1f} Hz; "
      f"folder proxies per class: {inv.groupby('class10', sort=False).size().to_dict()}")
display(inv.round(1))
display(pd.crosstab(folds.class10, folds.dev_fold).reindex(CLASS10))
folds[folds.class10 != "normal"].groupby(["dev_fold"]).folder.apply(lambda s: ", ".join(sorted(set(s)))).to_frame()

**Limitations.**
- Speeds below 40 Hz only (the speed-block test is reserved).
- Few folder proxies per class: 3 for seven of the nine fault classes, one for normal.
- This exercise uses examples of every class that is later withheld in LOCO. LOCO runs can therefore demonstrate
  fold-specific exclusion during fitting and calibration, but cannot claim those classes never influenced overall method
  development. Fold-safe thresholds alone do not resolve this.

## 2. Results per model (recording level)
Out-of-fold: each model is fitted K times, on the other folds, with scalers and SVM calibration refitted inside each
fold; every pool recording is scored exactly once. Intervals: stratified, mixed-unit bootstrap conditional on the fitted
folds (fault folder proxies resampled within class with all their recordings; normal recordings individually). The
number of folder proxies per class is shown with the table; **between-session uncertainty for normal cannot be estimated
(one folder)**.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Builds the per-model table: recording-level macro-F1 and balanced accuracy with their bootstrap intervals, for class10 and class6, window-level macro-F1 for contrast, and the total fitting time over the K folds.</div>

In [ ]:
order = ["lgbm", "rf", "logreg", "svm", "knn", "ensemble"]
fmt = lambda v, ci: f"{v:.3f} [{ci[0]:.3f}, {ci[1]:.3f}]"
rows = []
for m in order:
    r = dev["models"][m]
    rows.append({"model": m,
                 "class10 macro-F1": fmt(r["class10"]["macro_f1"], r["class10"]["macro_f1_ci"]),
                 "class10 balanced acc": fmt(r["class10"]["balanced_accuracy"], r["class10"]["balanced_accuracy_ci"]),
                 "class6 macro-F1": fmt(r["class6"]["macro_f1"], r["class6"]["macro_f1_ci"]),
                 "class6 balanced acc": fmt(r["class6"]["balanced_accuracy"], r["class6"]["balanced_accuracy_ci"]),
                 "window macro-F1": round(r["window_macro_f1"], 3),
                 "fit s (K folds)": round(r.get("seconds_total", np.nan), 1)})
table = pd.DataFrame(rows).set_index("model")
table.to_csv(RES / "dev_eval_table.csv")
print("folder proxies per class:", dev["provenance"]["folder_proxies_per_class"])
table

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows each model's F1 for every class (class10), on all pool recordings scored out-of-fold.</div>

In [ ]:
pd.DataFrame({m: dev["models"][m]["class10"]["per_class_f1"] for m in order}).loc[list(CLASS10)].round(3)

### The hardest folder proxies
Share of each folder's recordings classified correctly (class10), per model, sorted by the mean over models.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Tabulates, for every folder proxy, the share of its recordings each model classifies correctly, and lists the folders with the lowest average.</div>

In [ ]:
acc = pd.DataFrame({m: {f: v["accuracy"] for f, v in dev["models"][m]["class10"]["per_folder_accuracy"].items()}
                    for m in order})
acc["n"] = pd.Series({f: v["n"] for f, v in dev["models"]["lgbm"]["class10"]["per_folder_accuracy"].items()})
acc["mean over models"] = acc[order].mean(axis=1)
hard = acc.sort_values("mean over models")
hard.head(8).round(3)

Where do the recordings of the hardest folders go? Recording-level predictions of LightGBM and SVM:

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Rebuilds recording-level predictions from the saved out-of-fold window scores (same aggregation as everywhere) and counts, for the four hardest folders, which class their recordings were predicted as.</div>

In [ ]:
from ml.evaluation import aggregate_recordings

pool_f = folds.set_index("rec_id").folder


def rec_preds(model: str) -> pd.DataFrame:
    s = pd.read_parquet(SCORES / f"dev_r1_{model}.parquet").sort_values(["rec_id", "window_idx"])
    agg = aggregate_recordings(s[[f"score_{c}" for c in CLASS10]].to_numpy(), s.rec_id.to_numpy(),
                               cfg.baselines.prob_floor)
    return pd.DataFrame({"pred": agg.idxmax(axis=1), "folder": pool_f.loc[agg.index]})


worst = list(hard.index[:4])
pd.concat({m: rec_preds(m).query("folder in @worst").groupby("folder").pred.value_counts().unstack(fill_value=0)
           for m in ["lgbm", "svm"]}, names=["model"]).fillna(0).astype(int)

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Draws class10 confusion matrices for LightGBM and SVM over all pool recordings scored out-of-fold.</div>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, m in zip(axes, ["lgbm", "svm"]):
    cm = np.array(dev["models"][m]["class10"]["confusion"])
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(10), CLASS10, rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(10), CLASS10, fontsize=7)
    for (a, b), v in np.ndenumerate(cm):
        if v:
            ax.text(b, a, v, ha="center", va="center", fontsize=7, color="white" if v > cm.max() / 2 else "black")
    ax.set(title=f"{m}, class10, folder-grouped out-of-fold", xlabel="predicted", ylabel="true")
    ax.grid(False)
fig.tight_layout()
fig.savefig(FIG / "05b_confusion_dev.png")

## Findings (per-model results)
- **Folder-grouped out-of-fold evaluation is not saturated.** Recording-level class10 macro-F1 (804 recordings, K = 3, 95 % mixed-unit interval): SVM 0.943 [0.896, 0.980], ensemble 0.943 [0.894, 0.982], logistic regression 0.941 [0.893, 0.982], k-NN 0.930 [0.881, 0.973], random forest 0.905 [0.831, 0.967], LightGBM 0.898 [0.814, 0.963]. class6: 0.933–0.981. Intervals overlap widely; with 3 folder proxies for most fault classes they are wide, and they say nothing about between-session variation for normal (one folder).
- **The errors concentrate in the 20 g bearing folders**, the largest added mass left in the pool: overhang_outer/20g (mean correct over models 0.40), underhang_cage/20g (0.51), overhang_cage/20g (0.53). Their recordings are confused with other bearing classes, cage with outer race and across bearing positions (e.g. LightGBM predicts overhang_outer for 19 of 25 underhang_cage/20g recordings), not with normal or the shaft faults. For LightGBM and random forest, horizontal_misalignment/1.5mm (0.62, 0.57) is also hard.
- The simpler scaled models (logistic regression, SVM) do at least as well as the tree ensembles here, the reverse of what a saturated split can show; the differences are within the intervals.
